# LeetCode #1203: Sort Items by Groups Respecting Dependencies

https://leetcode.com/problems/sort-items-by-groups-respecting-dependencies/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n! \cdot (n + m))$ | $O(n)$ |
| **Optimal: Two-Level Topological Sort ★** | $O(n + m)$ | $O(n + m)$ |

---

## Understanding the Methods

### Brute Force
Try every permutation of items, check dependency and group-contiguity constraints. Completely intractable for $n > 10$.

### Optimal: Two-Level Topological Sort ★
Assign each ungrouped item (group = -1) a unique group id. Build two DAGs: one for inter-group dependencies, one for intra-group item dependencies. Topologically sort both; if either has a cycle, return []. Finally, emit items group by group, in group-topological order.

**Constraints:**
* $1 \le n \le 3 \times 10^4$
* $0 \le m \le 3 \times 10^4$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;
using System.Linq;

public class Solution {
    public int[] SortItems(int n, int m, int[] group, IList<IList<int>> beforeItems) {
        // Give every ungrouped item its own unique group
        int groupCount = m;
        for (int i = 0; i < n; i++)
            if (group[i] == -1) group[i] = groupCount++;

        // Adjacency lists and in-degrees for items and groups
        var itemAdj  = new List<int>[n];
        var groupAdj = new List<int>[groupCount];
        var itemIn   = new int[n];
        var groupIn  = new int[groupCount];
        for (int i = 0; i < n; i++) itemAdj[i] = new List<int>();
        for (int i = 0; i < groupCount; i++) groupAdj[i] = new List<int>();

        foreach (int item in Enumerable.Range(0, n)) {
            foreach (int pre in beforeItems[item]) {
                itemAdj[pre].Add(item);
                itemIn[item]++;
                // Inter-group edge (avoid self-loops and duplicates via set)
                if (group[pre] != group[item]) {
                    groupAdj[group[pre]].Add(group[item]);
                    groupIn[group[item]]++;
                }
            }
        }

        // Topological sort helper (Kahn's)
        int[] TopoSort(int[] inDeg, List<int>[] adj, int total) {
            var q = new Queue<int>();
            for (int i = 0; i < total; i++) if (inDeg[i] == 0) q.Enqueue(i);
            var order = new List<int>();
            while (q.Count > 0) {
                int u = q.Dequeue(); order.Add(u);
                foreach (int v in adj[u])
                    if (--inDeg[v] == 0) q.Enqueue(v);
            }
            return order.Count == total ? order.ToArray() : null; // null = cycle
        }

        var itemOrder  = TopoSort(itemIn, itemAdj, n);
        var groupOrder = TopoSort(groupIn, groupAdj, groupCount);
        if (itemOrder == null || groupOrder == null) return new int[0];

        // Bucket items by group in item-topological order
        var groupItems = new List<int>[groupCount];
        for (int i = 0; i < groupCount; i++) groupItems[i] = new List<int>();
        foreach (int item in itemOrder) groupItems[group[item]].Add(item);

        // Emit all items in group-topological order
        var result = new List<int>();
        foreach (int g in groupOrder)
            result.AddRange(groupItems[g]);
        return result.ToArray();
    }
}

### Python

In [ ]:
from collections import deque, defaultdict

class Solution:
    def sortItems(self, n: int, m: int, group: list[int], beforeItems: list[list[int]]) -> list[int]:
        # Give each ungrouped item its own synthetic group
        group_count = m
        for i in range(n):
            if group[i] == -1:
                group[i] = group_count
                group_count += 1

        item_adj   = [[] for _ in range(n)]
        group_adj  = [[] for _ in range(group_count)]
        item_in    = [0] * n
        group_in   = [0] * group_count

        for item in range(n):
            for pre in beforeItems[item]:
                item_adj[pre].append(item)
                item_in[item] += 1
                if group[pre] != group[item]:
                    group_adj[group[pre]].append(group[item])
                    group_in[group[item]] += 1

        def topo_sort(in_deg, adj, total):
            q = deque(i for i in range(total) if in_deg[i] == 0)
            order = []
            while q:
                u = q.popleft()
                order.append(u)
                for v in adj[u]:
                    in_deg[v] -= 1
                    if in_deg[v] == 0:
                        q.append(v)
            return order if len(order) == total else None  # None = cycle

        item_order  = topo_sort(item_in, item_adj, n)
        group_order = topo_sort(group_in, group_adj, group_count)
        if item_order is None or group_order is None:
            return []

        # Bucket each item into its group, preserving item-topo order
        group_items = defaultdict(list)
        for item in item_order:
            group_items[group[item]].append(item)

        # Concatenate groups in group-topological order
        return [item for g in group_order for item in group_items[g]]

### Go

In [ ]:
func sortItems(n int, m int, group []int, beforeItems [][]int) []int {
	groupCount := m
	for i := 0; i < n; i++ {
		if group[i] == -1 {
			group[i] = groupCount
			groupCount++
		}
	}

	itemAdj  := make([][]int, n)
	groupAdj := make([][]int, groupCount)
	itemIn   := make([]int, n)
	groupIn  := make([]int, groupCount)

	for item := 0; item < n; item++ {
		for _, pre := range beforeItems[item] {
			itemAdj[pre] = append(itemAdj[pre], item)
			itemIn[item]++
			if group[pre] != group[item] {
				groupAdj[group[pre]] = append(groupAdj[group[pre]], group[item])
				groupIn[group[item]]++
			}
		}
	}

	topoSort := func(inDeg []int, adj [][]int, total int) []int {
		q := []int{}
		for i := 0; i < total; i++ {
			if inDeg[i] == 0 { q = append(q, i) }
		}
		order := []int{}
		for len(q) > 0 {
			u := q[0]; q = q[1:]
			order = append(order, u)
			for _, v := range adj[u] {
				inDeg[v]--
				if inDeg[v] == 0 { q = append(q, v) }
			}
		}
		if len(order) == total { return order }
		return nil // cycle
	}

	itemOrder  := topoSort(itemIn, itemAdj, n)
	groupOrder := topoSort(groupIn, groupAdj, groupCount)
	if itemOrder == nil || groupOrder == nil { return nil }

	groupItems := make([][]int, groupCount)
	for _, item := range itemOrder {
		groupItems[group[item]] = append(groupItems[group[item]], item)
	}
	result := []int{}
	for _, g := range groupOrder {
		result = append(result, groupItems[g]...)
	}
	return result
}

### Rust

In [ ]:
use std::collections::VecDeque;

impl Solution {
    pub fn sort_items(n: i32, m: i32, mut group: Vec<i32>, before_items: Vec<Vec<i32>>) -> Vec<i32> {
        let n = n as usize;
        let mut group_count = m as usize;

        // Assign each ungrouped item its own group
        for i in 0..n {
            if group[i] == -1 {
                group[i] = group_count as i32;
                group_count += 1;
            }
        }

        let mut item_adj  = vec![vec![]; n];
        let mut group_adj = vec![vec![]; group_count];
        let mut item_in   = vec![0i32; n];
        let mut group_in  = vec![0i32; group_count];

        for item in 0..n {
            for &pre in &before_items[item] {
                let pre = pre as usize;
                item_adj[pre].push(item);
                item_in[item] += 1;
                if group[pre] != group[item] {
                    group_adj[group[pre] as usize].push(group[item] as usize);
                    group_in[group[item] as usize] += 1;
                }
            }
        }

        let topo = |in_deg: &mut Vec<i32>, adj: &Vec<Vec<usize>>, total: usize| -> Option<Vec<usize>> {
            let mut q: VecDeque<usize> = (0..total).filter(|&i| in_deg[i] == 0).collect();
            let mut order = vec![];
            while let Some(u) = q.pop_front() {
                order.push(u);
                for &v in &adj[u] {
                    in_deg[v] -= 1;
                    if in_deg[v] == 0 { q.push_back(v); }
                }
            }
            if order.len() == total { Some(order) } else { None }
        };

        let item_order  = topo(&mut item_in,  &item_adj,  n)?;
        let group_order = topo(&mut group_in, &group_adj, group_count)?;

        let mut group_items: Vec<Vec<i32>> = vec![vec![]; group_count];
        for item in item_order {
            group_items[group[item] as usize].push(item as i32);
        }
        Some(group_order.iter().flat_map(|&g| group_items[g].iter().copied()).collect())
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=8, m=2, group=[-1,-1,1,0,0,1,0,-1], beforeItems=[[],[6],[5],[6],[3,6],[],[],[]]`
Items 3,4,6 are in group 0; 2,5 in group 1; 0,1,7 each get their own synthetic group. The two-level sort produces a valid order such as `[6,3,4,1,5,2,0,7]`.

### 2. Slightly Complex
**Input:** `n=5, m=3, group=[0,0,2,1,0], beforeItems=[[3],[],[],[],[1,2]]`
Item 4 depends on items 1 and 2, which are in different groups from 4. Inter-group edges force group 1 and group 2 before group 0. Answer: a valid topological interleaving.

### 3. Edge Case: Time Factor
**Input:** $n=30000$ items with a chain of dependencies spanning all groups.
Both topological sorts run in $O(n + m)$. The bottleneck is building the adjacency lists — still linear in total edge count.

### 4. Edge Case: Space Factor
**Input:** Every item has its own group (all $group[i] = -1$); 30 000 inter-item dependencies.
Group count grows to $2n$. Both adjacency lists hold $O(n + m)$ entries total.

### 5. Almost-Impossible but Plausible
**Input:** Items 0→1 and 1→0 (a cycle within the same group).
Item-level topological sort detects the cycle and returns an empty list. The inter-group sort may succeed but the result is `[]` because the inner DAG is invalid.